In [2]:
pip install xgboost

  Obtaining dependency information for xgboost from https://files.pythonhosted.org/packages/5e/03/15cd49e855c62226ecf1831bbe4c8e73a4324856077a23c495538a36e557/xgboost-3.0.0-py3-none-win_amd64.whl.metadata
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB 1.3 MB/s eta 0:01:58
   ---------------------------------------- 0.3/150.0 MB 3.9 MB/s eta 0:00:39
   ---------------------------------------- 1.0/150.0 MB 7.9 MB/s eta 0:00:19
    --------------------------------------- 1.9/150.0 MB 12.1 MB/s eta 0:00:13
    --------------------------------------- 2.9/150.0 MB 14.2 MB/s eta 0:00:11
   - -------------------------------------- 3.8/150.0 MB 15.3 MB/s eta 0:00:10
   - -------------------------------------- 4.7/150.0 MB 15.9 MB/s eta 0:00:10
   - -------------------------------------- 5.8/150.0 MB 16.8 MB/s eta 0:00:09
   - -------------------------------------- 6.6/150.0 MB 16.7 MB/s eta 0:00:09
   -- ----------

In [27]:
# Step 1: Import Libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [30]:
# Load the CSV file
csv_path = '../NeoJaundice/chd_jaundice.csv'
df = pd.read_csv(csv_path)

# Extract metadata and target variable
metadata = df[['patient_id', 'gender', 'gestational_age', 'age(day)', 'weight']]
target = df['blood(mg/dL)']


# One-hot encode the 'gender' column
encoder = OneHotEncoder(sparse_output=False, drop='first')
gender_encoded = encoder.fit_transform(metadata[['gender']])

# Separate numeric columns for imputation
numeric_columns = ['gestational_age', 'age(day)', 'weight']
numeric_metadata = metadata[numeric_columns]

# Impute missing values in numeric columns
imputer = SimpleImputer(strategy='mean')
numeric_metadata_imputed = imputer.fit_transform(numeric_metadata)

# Combine imputed numeric metadata with one-hot encoded 'gender'
metadata_encoded = np.hstack((numeric_metadata_imputed, gender_encoded))

In [31]:
# Load and preprocess images
def load_and_preprocess_images(image_folder, image_names, target_size=(224, 224)):
    images = []
    for img_name in image_names:
        img_path = os.path.join(image_folder, img_name)
        img = image.load_img(img_path, target_size=target_size)
        img = image.img_to_array(img) / 255.0  # Normalize to [0, 1]
        images.append(img)
    return np.array(images)

image_folder = "../NeoJaundice/images"
image_names = df['image_idx']
images = load_and_preprocess_images(image_folder, image_names)

In [32]:
def build_cnn(input_shape):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
    ])
    return model

# Build the CNN
input_shape = images[0].shape
cnn_model = build_cnn(input_shape)
cnn_model.summary()

C:\Users\DELL\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │    11,075,712 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,168,960 (42.61 MB)

 Trainable params: 11,168,960 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
# Extract feature vectors
feature_vectors = cnn_model.predict(images)

# Combine metadata with feature vectors
combined_features = np.hstack((feature_vectors, metadata_encoded))

# Step 4: Handle Missing Values in Target Variable
# Drop rows with missing values in the target variable
target = target.dropna()
combined_features = combined_features[~np.isnan(combined_features).any(axis=1)]

70/70 ━━━━━━━━━━━━━━━━━━━━ 8s 112ms/step


In [34]:
# Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(combined_features, target, test_size=0.2, random_state=42)

In [35]:
# Step 6: Train Standard Regressors
regressors = {
    "Linear Regression": LinearRegression(),
    "LASSO": Lasso(alpha=0.01),
    "k-NN": KNeighborsRegressor(n_neighbors=5),
    "SVR": SVR(kernel='rbf'),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "ADA Boost": AdaBoostRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
}

In [36]:
results = {}
for name, regressor in regressors.items():
    # Train
    regressor.fit(X_train, y_train)
    
    # Predict
    y_pred = regressor.predict(X_test)
    
    # Evaluate
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {"MSE": mse, "R2": r2}
    print(f"{name}: MSE = {mse:.4f}, R2 = {r2:.4f}")

Linear Regression: MSE = 26.2580, R2 = 0.1056
LASSO: MSE = 27.7925, R2 = 0.0534
k-NN: MSE = 17.4862, R2 = 0.4044
SVR: MSE = 28.7623, R2 = 0.0203
Decision Tree: MSE = 26.5808, R2 = 0.0946
Random Forest: MSE = 12.5906, R2 = 0.5712
ADA Boost: MSE = 17.8695, R2 = 0.3913
Gradient Boosting: MSE = 13.5585, R2 = 0.5382
XGBoost: MSE = 13.3696, R2 = 0.5446
